# Will This Visitor Buy? Real-Time Purchase Intent
**NextGen Institute · Machine Learning · Final Project**
*Notebook version: v2 (dense-output fix)*

## 1. Problem framing
- **Who has the problem:** an online store where most visitors leave without buying.
- **Decision the model supports:** during a session, decide the on-site action: *no action*, *show a coupon*, or *show a chat assistant*. This avoids giving discounts to everyone.
- **Target:** `Revenue` (True = the session ended in a purchase).
- **Metric that matters:** the classes are imbalanced, so **accuracy is misleading**. We track **ROC-AUC** and **PR-AUC** to compare models, and **recall / precision / F1** at a chosen threshold. Missing a real buyer costs a sale; an unneeded coupon costs margin.

**Dataset:** Online Shoppers Purchasing Intention (UCI) · https://archive.ics.uci.edu/dataset/468

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib, sklearn
from scipy.stats import loguniform

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay, PrecisionRecallDisplay,
                             precision_recall_curve)
from sklearn.inspection import permutation_importance

from features import add_features, NUM_COLS, CAT_COLS, RAW_FEATURES, MONTH_ORDER

RANDOM_STATE = 42
DATA_PATH = "online_shoppers_intention.csv"   # download from the UCI link above
sns.set_theme(style="whitegrid")
print("scikit-learn", sklearn.__version__)

## 2. Data understanding
Source, size, column dictionary, types, missing values and class balance.

| Column | Meaning |
|---|---|
| Administrative / Informational / ProductRelated (+ `_Duration`) | pages visited per type and seconds spent on them |
| BounceRates / ExitRates | Google Analytics bounce and exit rates of visited pages |
| PageValues | average value of pages visited before a transaction |
| SpecialDay | closeness to a special day (e.g. Valentine's) from 0 to 1 |
| Month, Weekend | when the session happened |
| OperatingSystems, Browser, Region, TrafficType | technical / source codes (treated as **categories**) |
| VisitorType | New / Returning / Other |
| **Revenue** | **target**: purchase made or not |

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
n_dup = df.duplicated().sum()
print("Duplicate rows:", n_dup)
df = df.drop_duplicates().reset_index(drop=True)   # identical rows could land in both train and test
print("Shape after dropping duplicates:", df.shape)
display(df.head())
print(df.dtypes)

In [ ]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "none")
print("\nClass balance (Revenue):")
print(df["Revenue"].value_counts())
print(df["Revenue"].value_counts(normalize=True).round(3))
print("\nNegative values in duration columns (data quirk):",
      {c: int((df[c] < 0).sum()) for c in ["Administrative_Duration","Informational_Duration","ProductRelated_Duration"]})
df.describe().T

## 3. EDA
Univariate, bivariate and multivariate analysis.

In [ ]:
# Univariate: target + numeric distributions
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x="Revenue", data=df, ax=ax[0]); ax[0].set_title("Target balance")
sns.countplot(x="Month", data=df, order=[m for m in MONTH_ORDER if m in df["Month"].unique()], ax=ax[1])
ax[1].set_title("Sessions per month"); plt.tight_layout(); plt.show()

num_raw = ["Administrative","Informational","ProductRelated","ProductRelated_Duration",
           "BounceRates","ExitRates","PageValues","SpecialDay"]
df[num_raw].hist(bins=40, figsize=(12, 6)); plt.tight_layout(); plt.show()

In [ ]:
# Bivariate: conversion rate by group, and numeric features vs target
conv = lambda col: df.groupby(col)["Revenue"].mean().sort_values(ascending=False)
print("Conversion by VisitorType:\n", conv("VisitorType").round(3), "\n")
print("Conversion by Weekend:\n", conv("Weekend").round(3), "\n")
month_conv = df.groupby("Month")["Revenue"].mean().reindex([m for m in MONTH_ORDER if m in df["Month"].unique()])
print("Conversion by Month:\n", month_conv.round(3))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(x=month_conv.index, y=month_conv.values, ax=ax[0]); ax[0].set_title("Conversion rate by month")
sns.barplot(x="VisitorType", y="Revenue", data=df, ax=ax[1]); ax[1].set_title("Conversion rate by visitor type")
sns.boxplot(x="Revenue", y="PageValues", data=df, ax=ax[2]); ax[2].set_yscale("symlog"); ax[2].set_title("PageValues vs purchase")
plt.tight_layout(); plt.show()

print(df.groupby("Revenue")[["ExitRates","BounceRates","ProductRelated","PageValues"]].mean().round(3))

In [ ]:
# Multivariate: correlation heatmap of numeric features + target
corr_cols = ["Administrative","Administrative_Duration","Informational","Informational_Duration",
             "ProductRelated","ProductRelated_Duration","BounceRates","ExitRates","PageValues","SpecialDay"]
corr = df[corr_cols].assign(Revenue=df["Revenue"].astype(int)).corr()
plt.figure(figsize=(9, 7)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.show()
print("Correlation with Revenue:\n", corr["Revenue"].drop("Revenue").sort_values(ascending=False).round(3))

### EDA insights (fill in from the outputs above)
Write 3-5 short insights **using the numbers printed above**, e.g.:
1. *Class balance:* only ____% of sessions end in a purchase, so accuracy alone is misleading.
2. *PageValues:* purchasing sessions have a much higher PageValues (mean ____ vs ____).
3. *Exit/bounce rates:* buyers have lower ExitRates (____ vs ____).
4. *Visitor type:* new vs returning conversion is ____% vs ____%.
5. *Seasonality:* the strongest month is ____ and the weakest is ____.

## 4. Preprocessing & pipeline (no leakage)
The data is **split first**. Every transformation (imputation, scaling, encoding) lives inside a scikit-learn `Pipeline`, so it is fitted on training folds only. Row-wise feature engineering (`TotalPages`, `TotalDuration`, `AvgTimePerPage`, `HasPageValue`, log of durations) lives in `features.py`. It learns nothing from the data, and the same code is reused by the web app.

In [ ]:
X = df[RAW_FEATURES]
y = df["Revenue"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Purchase rate train / test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

def make_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_COLS),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT_COLS),
    ], sparse_threshold=0)   # dense output: HistGradientBoosting does not accept sparse input

def make_pipe(model):
    return Pipeline([("features", FunctionTransformer(add_features)),
                     ("prep", make_preprocess()),
                     ("model", model)])

In [ ]:
# Quick check (seconds): fails fast if the pipeline output is sparse or Gradient Boosting cannot fit
from scipy.sparse import issparse
_Xs, _ys = X_train.head(2000), y_train.head(2000)
_out = make_preprocess().fit_transform(add_features(_Xs), _ys)
print("Preprocess output is dense:", not issparse(_out), "| shape:", _out.shape)
make_pipe(HistGradientBoostingClassifier(max_iter=20)).fit(_Xs, _ys)
print("Quick pipeline check passed (notebook version v2)")

## 5. Baseline + 6. Modeling, tuning & cross-validation
- **Baseline:** always predict the majority class.
- **Models:** Logistic Regression, Random Forest, Gradient Boosting (`HistGradientBoostingClassifier`).
- **Tuning:** `RandomizedSearchCV` with stratified 5-fold CV on the **training set only**, scored by ROC-AUC.

In [ ]:
def metrics_at(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"roc_auc": roc_auc_score(y_true, proba),
            "pr_auc": average_precision_score(y_true, proba),
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred)}

baseline = make_pipe(DummyClassifier(strategy="most_frequent")).fit(X_train, y_train)
base_metrics = metrics_at(y_test, baseline.predict_proba(X_test)[:, 1])
print("Baseline (majority class):", {k: round(v, 3) for k, v in base_metrics.items()})

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search_space = {
    "Logistic Regression": (
        LogisticRegression(max_iter=3000, class_weight="balanced"),
        {"model__C": loguniform(1e-2, 1e2)}),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
        {"model__n_estimators": [200, 300], "model__max_depth": [5, 8, 12, None],
         "model__min_samples_leaf": [1, 3, 5, 10]}),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {"model__learning_rate": [0.03, 0.05, 0.1], "model__max_depth": [3, 5, None],
         "model__max_leaf_nodes": [15, 31, 63], "model__l2_regularization": [0.0, 0.1, 1.0],
         "model__min_samples_leaf": [20, 50]}),
}

fitted, cv_scores = {}, {}
for name, (model, params) in search_space.items():
    rs = RandomizedSearchCV(make_pipe(model), params, n_iter=8, scoring="roc_auc",
                            cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True)
    rs.fit(X_train, y_train)
    fitted[name], cv_scores[name] = rs.best_estimator_, rs.best_score_
    print(f"{name:22s} CV ROC-AUC = {rs.best_score_:.4f}  best params = {rs.best_params_}")

## 7. Evaluation on the untouched test set
The test set is used **once** here. The final model is chosen by **cross-validation score**, not by test score.

In [ ]:
rows = [{"model": "Baseline (majority class)", "cv_roc_auc": np.nan, **base_metrics}]
test_proba = {}
for name, est in fitted.items():
    p = est.predict_proba(X_test)[:, 1]
    test_proba[name] = p
    rows.append({"model": name, "cv_roc_auc": cv_scores[name], **metrics_at(y_test, p)})
results = pd.DataFrame(rows).set_index("model").round(3)
display(results)

best_name = max(cv_scores, key=cv_scores.get)
best_model = fitted[best_name]
print("Selected by CV ROC-AUC:", best_name)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in test_proba.items():
    RocCurveDisplay.from_predictions(y_test, p, name=name, ax=ax[0])
    PrecisionRecallDisplay.from_predictions(y_test, p, name=name, ax=ax[1])
ax[0].set_title("ROC curves (test)"); ax[1].set_title("Precision-Recall curves (test)")
plt.tight_layout(); plt.show()

### Choosing the decision threshold
The default 0.5 is rarely right for imbalanced data. Using **out-of-fold predictions on the training set only**, we derive two thresholds for the site's actions:
- `t_chat` = threshold that maximizes **F1** → above it the visitor is likely to buy → **show chat** (help close the sale, no discount).
- `t_coupon` = lowest threshold that still keeps **recall ≥ 90%** → between `t_coupon` and `t_chat` the visitor is on the fence → **show coupon**. Below `t_coupon` → **no action**.

These cut-offs are a business policy. Adjust the 90% target to the company's coupon budget.

In [ ]:
oof = cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
t_chat = float(thr[np.argmax(f1s)])
ok = thr[rec[:-1] >= 0.90]
t_coupon = float(ok.max()) if len(ok) else t_chat / 2
if t_coupon >= t_chat:
    t_coupon = t_chat / 2
print(f"t_chat (max F1 on OOF) = {t_chat:.3f} | t_coupon (recall>=90% on OOF) = {t_coupon:.3f}")

p_best = test_proba[best_name]
for label, t in [("default 0.5", 0.5), ("t_chat (F1-opt)", t_chat), ("t_coupon (recall>=90%)", t_coupon)]:
    m = metrics_at(y_test, p_best, t)
    print(f"{label:24s} precision={m['precision']:.3f} recall={m['recall']:.3f} f1={m['f1']:.3f}")

pred_final = (p_best >= t_chat).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final), display_labels=["No buy", "Buy"]).plot(cmap="Blues")
plt.title(f"{best_name} - confusion matrix @ t_chat"); plt.show()
print(classification_report(y_test, pred_final, target_names=["No buy", "Buy"]))

### Error analysis
Where does the model miss real buyers (false negatives) or raise false alarms (false positives)?

In [ ]:
err = X_test.copy()
err["y"] = y_test.values; err["pred"] = pred_final; err["proba"] = p_best
err["FN"] = ((err.y == 1) & (err.pred == 0)).astype(int)
err["FP"] = ((err.y == 0) & (err.pred == 1)).astype(int)
print("False negatives:", int(err.FN.sum()), "| False positives:", int(err.FP.sum()))

print("\nMiss rate among real buyers, by VisitorType:")
print(err[err.y == 1].groupby("VisitorType")["FN"].agg(["mean", "count"]).round(3))
print("\nMiss rate among real buyers, by PageValues > 0:")
print(err[err.y == 1].groupby(err["PageValues"] > 0)["FN"].agg(["mean", "count"]).round(3))
print("\nAverage features: missed buyers vs caught buyers")
print(err[err.y == 1].groupby("FN")[["PageValues", "ExitRates", "ProductRelated", "ProductRelated_Duration"]].mean().round(3))

# Lift: how many real buyers sit in the top-scored deciles?
lift = pd.DataFrame({"y": y_test.values, "p": p_best})
lift["decile"] = pd.qcut(lift["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
lt = lift.groupby("decile", observed=True)["y"].agg(["sum", "mean"]).rename(columns={"sum": "buyers", "mean": "purchase_rate"})
lt["share_of_all_buyers"] = lt["buyers"] / lt["buyers"].sum()
lt["cumulative_share"] = lt["share_of_all_buyers"].cumsum()
display(lt.round(3))

In [ ]:
# Which inputs matter most? (permutation importance on raw columns, test set)
pi = permutation_importance(best_model, X_test, y_test, scoring="roc_auc", n_repeats=5,
                            random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
imp.head(10).iloc[::-1].plot.barh(figsize=(7, 4), title="Permutation importance (drop in ROC-AUC)")
plt.tight_layout(); plt.show()
print(imp.head(10).round(4))

## 8. Business insight (fill in from your own results)
- **What the model does:** it scores every session with a purchase probability. At the F1-optimal threshold it reaches recall ____ and precision ____ (see tables above) versus a baseline recall of 0.
- **Recommended policy:**
  - `p >= t_chat` → open the **chat assistant** (they already intend to buy; don't discount).
  - `t_coupon <= p < t_chat` → show a **coupon** (on the fence).
  - `p < t_coupon` → **no action** (save the discount budget).
- **Targeting efficiency:** the top 10% of sessions by score contain ____% of all buyers (lift table).
- **Next step:** A/B test the policy to measure the real uplift in conversion and coupon ROI.

### Limitations
- Data comes from **one store over about one year**; seasonality and behaviour may not transfer.
- `PageValues` is computed from pages visited **before a transaction**, so in a truly real-time deployment it must be computed from the partial session only. Performance on partial sessions may be lower than reported here.
- Model scores are probabilities of purchase, **not causal uplift**: it does not prove that a coupon changes the decision.
- Duplicate rows were removed and codes such as `Region` and `TrafficType` have no documented meaning.

## 9. Save the pipeline for the web app

In [ ]:
def py(v):
    return v.item() if hasattr(v, "item") else v

cat_options = {c: sorted({py(v) for v in X_train[c].unique()}) for c in ["OperatingSystems", "Browser", "Region", "TrafficType"]}
defaults = {c: py(X_train[c].median()) for c in RAW_FEATURES if c not in CAT_COLS and c != "Weekend"}
defaults.update({c: py(X_train[c].mode()[0]) for c in CAT_COLS})
ranges = {c: py(X_train[c].quantile(0.99)) for c in RAW_FEATURES if c not in CAT_COLS and c != "Weekend"}
months = [m for m in MONTH_ORDER if m in set(X_train["Month"].unique())]

final = metrics_at(y_test, p_best, t_chat)
meta = {
    "pipeline": best_model,
    "model_name": best_name,
    "t_chat": t_chat, "t_coupon": t_coupon,
    "months": months,
    "visitor_types": sorted(X_train["VisitorType"].unique().tolist()),
    "cat_options": cat_options, "defaults": defaults, "ranges": ranges,
    "test_metrics_at_t_chat": {k: round(float(v), 4) for k, v in final.items()},
    "sklearn_version": sklearn.__version__,
    "n_train": int(len(X_train)),
}
joblib.dump(meta, "model.joblib", compress=3)
print("Saved model.joblib | model:", best_name, "| sklearn", sklearn.__version__)

# sanity check: reload and score one row exactly like the app does
reloaded = joblib.load("model.joblib")
print("Reload check, first test row ->", round(float(reloaded["pipeline"].predict_proba(X_test.iloc[[0]])[0, 1]), 4))

# results table for the README (copy-paste)
print("\n| Model | CV ROC-AUC | Test ROC-AUC | Test PR-AUC | Recall@0.5 | F1@0.5 |\n|---|---|---|---|---|---|")
for name, r in results.iterrows():
    cvv = "-" if pd.isna(r["cv_roc_auc"]) else f"{r['cv_roc_auc']:.3f}"
    print(f"| {name} | {cvv} | {r['roc_auc']:.3f} | {r['pr_auc']:.3f} | {r['recall']:.3f} | {r['f1']:.3f} |")
print(f"\nBest model @ t_chat={t_chat:.3f}: precision={final['precision']:.3f}, recall={final['recall']:.3f}, f1={final['f1']:.3f}")